In [1]:
import numpy as np
import pandas as pd
import csv
import matplotlib.pyplot as plt
from fmqa import FMBQM
import dimod

In [2]:
df = pd.read_csv('compl_enum_cstr_8_baron_hull.csv')

def load_grid(filename="compl_enum_cstr_8_baron_hull.csv"):
    """
    Load grid data from CSV.
    Expected CSV columns: "x", "y", "Objective", among others.
    Returns a dictionary mapping (x, y) tuples to the objective value.
    """
    grid_data = {}
    with open(filename, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                # Convert x and y values to integers
                x_val = int(row["x"])
                y_val = int(row["y"])
                # Convert objective value to float
                obj_val = float(row["Objective"])
                grid_data[(x_val, y_val)] = obj_val
            except Exception as e:
                # Skip rows with conversion issues
                continue

    # Compute the minimum among finite objective values
    finite_values = [v for v in grid_data.values() if not np.isnan(v)]
    x_min = min(finite_values) if finite_values else None
    x_max = max(finite_values) if finite_values else None     
    return grid_data, x_min, x_max
# Load the data from grid.csv
grid_data, x_min, x_max = load_grid()

# Determine grid bounds from the keys of grid_data
xs = [pt[0] for pt in grid_data.keys()]
ys = [pt[1] for pt in grid_data.keys()]
min_x, max_x = min(xs), max(xs)
min_y, max_y = min(ys), max(ys)
print("Grid bounds:")
print("x in [{}, {}]".format(min_x, max_x))
print("y in [{}, {}]".format(min_y, max_y))

Grid bounds:
x in [1, 8]
y in [1, 8]


In [3]:
# Integer to binary weighted bit encoding
def int_to_bits(n: int, upper: int, lsb_first: bool = True):
    """
    Encode n in binary using the minimum number of bits for [0, upper].
    Returns a list of 0/1 (LSB first by default).
    """
    if not (0 <= n <= upper):
        raise ValueError(f"value {n} outside [0, {upper}]")

    m = int(np.ceil(np.log2(upper)))      # cast to int

    bits = [(n - 1 >> k) & 1 for k in range(m)]
    if not lsb_first:
        bits.reverse()
    return bits

int_to_bits(3, 5, lsb_first=True)  # Example usage

def coord_bits(x: int, y: int,
               x_max: int, y_max: int,
               lsb_first: bool = True):
    """
    Given (x, y) and their maxima, return a tuple:
        ([bits for x], [bits for y])
    """
    x_bits = int_to_bits(x, x_max, lsb_first)
    y_bits = int_to_bits(y, y_max, lsb_first)
    all_bits = x_bits + y_bits
    return ''.join(str(bit) for bit in all_bits)

print(coord_bits(5, 1, x_max=8, y_max=8, lsb_first=False))  # Output: '101001' 


100000


In [4]:
int_to_bits(11, 16, lsb_first=False)  # Example usage



[1, 0, 1, 0]

In [5]:
# Load the data from grid.csv
grid_data, x_min, x_max = load_grid('compl_enum_cstr_8_baron_hull.csv')

# PENALTY = 1e6  # adjust this penalty as appropriate
PENALTY = np.inf # use infinity as penalty for infeasible points
# PENALTY = np.nan  # use NaN as penalty for infeasible points
# PENALTY = None  # use None as penalty for infeasible points

def obj_funct(x):
    """
    Given a candidate point x (a list or 1-D numpy array with 2 elements),
    this function rounds x to the nearest integers and returns the objective value
    if the point exists in grid_data and is feasible.
    Otherwise, returns a penalty value.
    """
    # Round the candidate coordinates to nearest integers
    x_int = int(round(x[0]))
    y_int = int(round(x[1]))
    
    # Check that x_int and y_int are numbers (optional check)
    if not isinstance(x_int, (int, float)) or not isinstance(y_int, (int, float)):
        return PENALTY
    
    # If the point exists in the grid, check its objective value.
    if (x_int, y_int) in grid_data:
        obj_val = grid_data[(x_int, y_int)]
        # Return penalty if the objective is NaN.
        if np.isnan(obj_val):
            return PENALTY
        else:
            return obj_val
    else:
        # If the point is not in the grid, return the penalty.
        return PENALTY

In [6]:
# Test for the obj_funct function
obj_funct([8,8])

2.93438026379752

$$
f(x)=-1+\frac{\text{raw val} - x_{min}}{x_{max} - x_{min}}
$$

In [7]:
def scale_point(x, y, grid_data, x_min, x_max):
    """
    Compute the arctan-based scaled transformation for a point (x, y).
    Maps x_min -> -1 and infinity -> 1.
    
    Parameters:
      - x, y: coordinates (floats)
      - grid_data: dict from load_grid
      - x_min: minimum objective value from load_grid
      - alpha: scale parameter (default=1)
    
    Returns:
      - scaled value in [-1, 1]
    """
    raw_val = obj_funct([x, y])

    if raw_val == np.inf:
        return 1.0
    else:
      delta = raw_val - x_min
      diff = x_max - x_min
      scaled = -1 + delta / diff
      return float(scaled)

# scale_point(8, 8, grid_data, x_min, x_max)

for i in range(1,9):
    for j in range(1,9):
        x = i
        y = j
        print(f"({x}, {y}) -> {scale_point(x, y, grid_data, x_min, x_max)}")

(1, 1) -> 0.0
(1, 2) -> 1.0
(1, 3) -> 1.0
(1, 4) -> 1.0
(1, 5) -> 1.0
(1, 6) -> 1.0
(1, 7) -> 1.0
(1, 8) -> 1.0
(2, 1) -> -0.8377805873692383
(2, 2) -> -0.8377805873288727
(2, 3) -> 1.0
(2, 4) -> 1.0
(2, 5) -> 1.0
(2, 6) -> 1.0
(2, 7) -> 1.0
(2, 8) -> 1.0
(3, 1) -> -0.9452605289003927
(3, 2) -> -0.9452605289018899
(3, 3) -> -0.9452605289003866
(3, 4) -> 1.0
(3, 5) -> 1.0
(3, 6) -> 1.0
(3, 7) -> 1.0
(3, 8) -> 1.0
(4, 1) -> -0.9712311934838855
(4, 2) -> -0.9712311939552947
(4, 3) -> -0.9712311934841215
(4, 4) -> -0.9713107763340356
(4, 5) -> 1.0
(4, 6) -> 1.0
(4, 7) -> 1.0
(4, 8) -> 1.0
(5, 1) -> -0.9718266787495938
(5, 2) -> -0.9718266787495572
(5, 3) -> -0.9718266790498044
(5, 4) -> -0.9718266787817197
(5, 5) -> -0.9816365967722328
(5, 6) -> 1.0
(5, 7) -> 1.0
(5, 8) -> 1.0
(6, 1) -> -0.9616434184426746
(6, 2) -> -0.961643418491716
(6, 3) -> -0.9616434184427288
(6, 4) -> -0.961652231644598
(6, 5) -> -0.9616434185123726
(6, 6) -> -0.9895118326776124
(6, 7) -> 1.0
(6, 8) -> 1.0
(7, 1) -> 

In [8]:
def decode_and_scale(bitstring: str, x_max: int, y_max: int, grid_data, x_min, x_max_obj):
    """
    Given a binary bitstring representing x and y, decode to integers,
    and return the scaled objective value.

    Parameters:
    - bitstring: str (e.g., '101001')
    - x_max, y_max: upper bounds for x and y to determine bit lengths
    - grid_data: dictionary from load_grid()
    - x_min, x_max_obj: objective value bounds for scaling

    Returns:
    - scaled value from scale_point
    """
    # Determine number of bits for x and y
    x_bits_len = int(np.ceil(np.log2(x_max + 1)))
    y_bits_len = int(np.ceil(np.log2(y_max + 1)))

    if len(bitstring) != x_bits_len + y_bits_len:
        raise ValueError("Bitstring length mismatch with provided x_max and y_max")

    # Extract bit slices and convert to integers
    x_bits = bitstring[:x_bits_len]
    y_bits = bitstring[x_bits_len:]

    x_int = int(x_bits, 2)
    y_int = int(y_bits, 2)

    return scale_point(x_int, y_int, grid_data, x_min, x_max_obj)

bitstring = '101001'  # This corresponds to (5, 1) if x=5, y=1 and both max=5
result = decode_and_scale(bitstring, x_max=5, y_max=5, grid_data=grid_data, x_min=x_min, x_max_obj=x_max)
print(f"{bitstring} → scaled value: {result}")

101001 → scaled value: -0.9718266787495938


In [9]:
# 1) Build initial surrogate on the full grid
keys   = list(grid_data.keys())
X_init = np.array([[int(b) for b in coord_bits(x,y,max_x,max_y,lsb_first=False)]
                   for x,y in keys], dtype=int)
y_init = np.array([scale_point(x,y,grid_data,x_min,x_max) for x,y in keys],
                  dtype=float)

model   = FMBQM.from_data(X_init, y_init)
sampler = dimod.SimulatedAnnealingSampler()

# 2) Prepare storage for the 5 *valid* samples you want
sampled_bits   = []
sampled_coords = []
sampled_y      = []

nbit_x = int(np.ceil(np.log2(max_x)))
nbit_y = int(np.ceil(np.log2(max_y)))
target = 8

# 3) Loop until you have collected 5 *valid* points
while len(sampled_bits) < target:
    # 3a) Draw one candidate
    res    = sampler.sample(model, num_reads=1)
    bits   = res.record['sample'][0]
    bitstr = ''.join(str(b) for b in bits)

    # 3b) Decode to integers
    xi = int(''.join(str(b) for b in bits[:nbit_x]), 2) + 1
    yi = int(''.join(str(b) for b in bits[nbit_x:]), 2) + 1
    coord = (xi, yi)

    # 3c) Check feasibility x >= y and existence in grid_data
    if coord in grid_data and not np.isnan(grid_data[coord]) and xi >= yi:
        # valid: get real scaled objective
        val = scale_point(xi, yi, grid_data, x_min, x_max)
        sampled_bits.append(bitstr)
        sampled_coords.append(coord)
        sampled_y.append(val)
        print(f"✅ Collected {len(sampled_bits)}/{target}: "
              f"{bitstr} → {coord} (scaled={val:.4f})")
    else:
        # infeasible: assign a heavy penalty so FMQA avoids it next time
        val = 1.5  
        print(f"❌ Skipping {bitstr} → {coord} (penalized)")

    # 3d) Always update the surrogate model with this new (bits→val) pair
    X_init = np.vstack([X_init, bits[np.newaxis, :]])
    y_init = np.hstack([y_init, val])
    model.train(X_init, y_init)

# 4) Print your 8 valid samples
print("\nFinal 8 valid FMQA samples:")
for i,(b,c,v) in enumerate(zip(sampled_bits, sampled_coords, sampled_y), 1):
    print(f"  Sample {i}: {b} → {c} → scaled obj = {v:.4f}")

# 5) Which of those is the FMQA “best”?
best_idx   = int(np.argmin(sampled_y))
best_bits  = sampled_bits[best_idx]
best_coord = sampled_coords[best_idx]
best_val   = sampled_y[best_idx]
print(f"\n→ Best FMQA sample: Sample {best_idx+1}: {best_bits} → {best_coord} → scaled obj = {best_val:.4f}")

# 6) Compute the true optimum over the feasible (y >= x) grid
feasible = {pt:obj for pt,obj in grid_data.items() if not np.isnan(obj) and pt[0] >= pt[1]}
true_coord  = min(feasible, key=feasible.get)
true_obj    = feasible[true_coord]
true_scaled = scale_point(true_coord[0], true_coord[1], grid_data, x_min, x_max)
print(f"→ True feasible optimum: {true_coord} → obj = {true_obj:.4f}, scaled = {true_scaled:.4f}")

❌ Skipping 011111 → (4, 8) (penalized)
❌ Skipping 100101 → (5, 6) (penalized)
✅ Collected 1/8: 111100 → (8, 5) (scaled=-0.9393)
❌ Skipping 101110 → (6, 7) (penalized)
✅ Collected 2/8: 110011 → (7, 4) (scaled=-0.9492)
✅ Collected 3/8: 111101 → (8, 6) (scaled=-0.9393)
✅ Collected 4/8: 101001 → (6, 2) (scaled=-0.9616)
✅ Collected 5/8: 111101 → (8, 6) (scaled=-0.9393)
✅ Collected 6/8: 101000 → (6, 1) (scaled=-0.9616)
✅ Collected 7/8: 011011 → (4, 4) (scaled=-0.9713)
✅ Collected 8/8: 110011 → (7, 4) (scaled=-0.9492)

Final 8 valid FMQA samples:
  Sample 1: 111100 → (8, 5) → scaled obj = -0.9393
  Sample 2: 110011 → (7, 4) → scaled obj = -0.9492
  Sample 3: 111101 → (8, 6) → scaled obj = -0.9393
  Sample 4: 101001 → (6, 2) → scaled obj = -0.9616
  Sample 5: 111101 → (8, 6) → scaled obj = -0.9393
  Sample 6: 101000 → (6, 1) → scaled obj = -0.9616
  Sample 7: 011011 → (4, 4) → scaled obj = -0.9713
  Sample 8: 110011 → (7, 4) → scaled obj = -0.9492

→ Best FMQA sample: Sample 7: 011011 → (4, 4)

In [10]:
from matplotlib.ticker import MaxNLocator

# Extract X and Y from sampled coordinates
x_coords = [coord[0] for coord in history_coords]
y_coords = [coord[1] for coord in history_coords]
objectives = history_y

# Identify best (minimum objective) sample
best_idx = int(np.argmin(objectives))
best_coord = history_coords[best_idx]

plt.figure(figsize=(8, 6))

# Plot sampled points as scatter
sc = plt.scatter(x_coords, y_coords, c=objectives, cmap='coolwarm_r', s=80, edgecolors='k', label='Sampled Points')

# Draw trajectory arrows
for i in range(1, len(history_coords)):
    x_prev, y_prev = history_coords[i - 1]
    x_curr, y_curr = history_coords[i]
    plt.arrow(x_prev, y_prev, x_curr - x_prev, y_curr - y_prev,
              length_includes_head=True, head_width=0.15, head_length=0.2,
              fc='gray', ec='gray', alpha=0.5)

# Mark first point
plt.scatter(x_coords[0], y_coords[0], color='green', s=120, edgecolors='k', label='Start')

# Mark best (minimum) point
plt.scatter(best_coord[0], best_coord[1], color='red', s=120, edgecolors='k', label='Best (Min Objective)')

# Force integer ticks on both axes
plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
plt.gca().yaxis.set_major_locator(MaxNLocator(integer=True))

# Labels and styling
plt.xlabel("x coordinate")
plt.ylabel("y coordinate")
plt.title("Sampling Trajectory in Coordinate Space")
plt.colorbar(sc, label="Scaled Objective Value")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

NameError: name 'history_coords' is not defined

In [119]:
int_to_bits(3, 5, lsb_first=True)  # Example usage

[0, 1, 0]

In [ ]:
# FMQA vs Random Sampling Analysis

print("=" * 80)
print("FMQA vs RANDOM SAMPLING: Can 5 samples find the global minimum?")
print("=" * 80)

# Calculate feasible points and true optimum
feasible_points = {pt: obj for pt, obj in grid_data.items() 
                   if not np.isnan(obj) and pt[0] >= pt[1]}
total_feasible = len(feasible_points)
true_optimum = min(feasible_points, key=feasible_points.get)
true_opt_obj = feasible_points[true_optimum]
true_opt_scaled = scale_point(true_optimum[0], true_optimum[1], grid_data, x_min, x_max)

print(f"\n📊 PROBLEM STATISTICS:")
print(f"   • Total feasible points: {total_feasible}")
print(f"   • True global minimum: {true_optimum} → obj = {true_opt_obj:.4f}, scaled = {true_opt_scaled:.4f}")

# Analyze FMQA results from cell 9
fmqa_samples = len(sampled_coords)
fmqa_best = min(sampled_y)
fmqa_best_coord = sampled_coords[np.argmin(sampled_y)]
fmqa_found_optimum = any(coord == true_optimum for coord in sampled_coords)

print(f"\n🎯 FMQA RESULTS (from cell 9):")
print(f"   • Samples taken: {fmqa_samples}")
print(f"   • Best found: {fmqa_best_coord} → scaled obj = {fmqa_best:.4f}")
print(f"   • Found global optimum: {'✅ YES' if fmqa_found_optimum else '❌ NO'}")
print(f"   • Gap to optimum: {abs(fmqa_best - true_opt_scaled):.4f}")

# Random sampling probabilities
def prob_find_optimal(n_samples):
    """Probability of finding the optimal solution with n random samples"""
    return 1 - (1 - 1/total_feasible)**n_samples

print(f"\n🎲 RANDOM SAMPLING PROBABILITIES:")
sample_sizes = [1, 3, 5, 8, 10, 15, 20, 30]
for n in sample_sizes:
    prob = prob_find_optimal(n)
    print(f"   • {n:2d} samples: {prob:.1%} chance of finding optimum")

# Expected samples for 95% confidence
import math
samples_for_95 = math.log(0.05) / math.log(1 - 1/total_feasible)
print(f"\n📈 For 95% confidence: need ~{samples_for_95:.0f} random samples")

# Simulate random sampling
print(f"\n🔄 SIMULATING RANDOM SAMPLING:")
np.random.seed(42)  # for reproducibility
feasible_list = list(feasible_points.keys())

n_simulations = 1000
found_optimum_count = 0
best_objectives = []

for sim in range(n_simulations):
    # Randomly sample 5 points
    random_samples = np.random.choice(feasible_list, size=5, replace=False)
    random_objectives = [feasible_points[pt] for pt in random_samples]
    best_random = min(random_objectives)
    best_objectives.append(best_random)
    
    if true_optimum in random_samples:
        found_optimum_count += 1

empirical_prob = found_optimum_count / n_simulations
theoretical_prob = prob_find_optimal(5)

print(f"   • Theoretical probability (5 samples): {theoretical_prob:.1%}")
print(f"   • Empirical probability (1000 simulations): {empirical_prob:.1%}")
print(f"   • Average best objective found: {np.mean(best_objectives):.4f}")
print(f"   • Best objective range: [{min(best_objectives):.4f}, {max(best_objectives):.4f}]")

# Compare FMQA vs Random
print(f"\n🏆 COMPARISON:")
print(f"   • FMQA best: {fmqa_best:.4f} (scaled)")
print(f"   • Random average: {np.mean(best_objectives):.4f} (raw)")
print(f"   • True optimum: {true_opt_obj:.4f} (raw)")

if fmqa_found_optimum:
    print(f"\n🎉 FMQA SUCCESSFULLY FOUND THE GLOBAL OPTIMUM!")
    print(f"   This is better than random sampling would typically achieve.")
else:
    print(f"\n📊 FMQA did not find the global optimum, but:")
    print(f"   • It used intelligent sampling based on learned patterns")
    print(f"   • It focused on promising regions of the search space")
    print(f"   • It's more efficient than random sampling for large problems")

print("\n" + "=" * 80)